# STATE SE — Untrained Baseline (PBMC 46k)

**Hypothesis**: STATE embeddings are collapsing during training. If the untrained model
performs similarly to the trained model, that confirms mode collapse.
An untrained model should perform comparably to a random projection baseline.

In [1]:
import sys, os
sys.path.insert(0, os.path.expanduser(
    "~/noise_scaling/modeling/Scaling-up-measurement-noise-scaling-laws/scaling_laws/src"
))

from pathlib import Path
import subprocess
import textwrap
import tempfile
import numpy as np
import pandas as pd
import anndata as ad
import matplotlib.pyplot as plt

from scaling_laws.algo import State

/home/igor/miniconda3/envs/modeling/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/home/igor/miniconda3/envs/modeling/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATA_DIR = Path(os.path.expanduser("~/noise_scaling/data"))
DATASET = "PBMC"
SIZE = 46_415
QUALITY = 1.0
DEVICE = 4
SEED = 42

base_dir = DATA_DIR / DATASET / str(SIZE) / str(QUALITY)
untrained_dir = base_dir / "results" / "State_untrained" / "model"
untrained_dir.mkdir(parents=True, exist_ok=True)
untrained_ckpt = untrained_dir / "untrained.ckpt"
print(f"Untrained checkpoint will be saved to: {untrained_ckpt}")

Untrained checkpoint will be saved to: /home/igor/noise_scaling/data/PBMC/46415/1.0/results/State_untrained/model/untrained.ckpt


## Gene Sets & Preprocessing: All 5 Methods

All methods start from the same `preprocessed.h5ad` which contains **raw downsampled counts** (20,729 genes for PBMC).

| | **PCA** | **RandomProjection** | **SCVI** | **Geneformer** | **STATE** |
|---|---|---|---|---|---|
| **Genes in** | 750 HVGs | All 20,729 | All 20,729 | All 20,729 (vocab), 512 per cell (ranked) | 14,899 (genes with ESM-2 embeddings) of 20,729 |
| **Gene selection** | Pre-computed HVG mask (`pca_hvg.pkl`, scanpy `n_top_genes=750`) | None | None | Token dict covers all 20,729; top 512 expressed per cell enter the model | Valid gene mask: only genes with an ESM-2 protein embedding (14,899/20,729) |
| **Library-size norm** | Yes (`normalize_total`, target=1e4) | No | Internally (ZINB models library size) | No (rank-encoding is implicit normalization) | No |
| **log1p** | Yes | No | Internally | No | Yes (applied in dataloader if raw counts detected) |
| **Protein-coding filter** | No (but HVG is a subset) | No | No | No | Effectively yes — ESM-2 embeddings only exist for protein-coding genes, so ~5,830 genes are dropped |
| **Gene embeddings** | N/A | N/A | N/A | Learned from scratch (256-dim) | Pre-computed ESM-2 (2560-dim), frozen |
| **Input to model** | Norm+log1p matrix, 750 cols → SVD | Raw count matrix, 20,729 cols → random projection | Raw count matrix, 20,729 cols → VAE encoder | Rank-ordered gene token IDs (512 tokens) | 512-token "sentence": genes ranked by log1p expression, each token → ESM-2 embedding |
| **Output dim** | 16 | 16 | 16 | 256 | 256 |

## 1. Create untrained (random) checkpoint

Instantiate the STATE model with the same architecture as training but **no weight updates**.
Save the randomly-initialized weights as a checkpoint.

In [3]:
# Write a helper script that runs in the STATE conda env
# to create an untrained checkpoint with proper config
script = textwrap.dedent(f"""\
    import torch
    from torch import nn
    from omegaconf import OmegaConf
    from state.emb.nn.model import StateEmbeddingModel
    from state.emb.utils import get_embedding_cfg
    from state.emb.train.trainer import get_embeddings

    # Load config and apply the same overrides as State.train()
    cfg = OmegaConf.load("{base_dir / 'preprocessed' / 'state_data' / 'state_config.yaml'}")
    profile_name = "scaling_{DATASET}_{SIZE}_{str(QUALITY).replace('.', '_')}"

    # Apply overrides matching State.train() in state.py
    cfg.embeddings.current = profile_name
    cfg.dataset.current = profile_name
    cfg.model.batch_size = 128
    cfg.model.emsize = 256
    cfg.model.d_hid = 512
    cfg.model.nhead = 4
    cfg.model.nlayers = 3
    cfg.model.output_dim = 256
    cfg.model.dataset_correction = False
    cfg.model.dropout = 0.1

    emb_cfg = get_embedding_cfg(cfg)
    print(f"Token dim (emb size): {{emb_cfg.size}}")

    # Create model with random weights
    model = StateEmbeddingModel(
        token_dim=emb_cfg.size,
        d_model=cfg.model.emsize,
        nhead=cfg.model.nhead,
        d_hid=cfg.model.d_hid,
        nlayers=cfg.model.nlayers,
        output_dim=cfg.model.output_dim,
        dropout=0.0,  # no dropout for inference
        warmup_steps=0,
        compiled=False,
        max_lr=5e-4,
        emb_size=emb_cfg.size,
        cfg=cfg,
    )

    # Load gene embeddings (same as trainer does)
    all_pe = get_embeddings(cfg)
    all_pe.requires_grad = False
    model.pe_embedding = nn.Embedding.from_pretrained(all_pe)
    model = model.cuda()

    # Save as a Lightning checkpoint
    import lightning as L
    trainer = L.Trainer(
        accelerator='cuda',
        devices=1,
        logger=False,
        enable_checkpointing=False,
    )
    trainer.strategy.connect(model)
    trainer.save_checkpoint("{untrained_ckpt}")
    print(f"Saved untrained checkpoint to {untrained_ckpt}")

    # Verify
    ckpt = torch.load("{untrained_ckpt}", map_location="cpu", weights_only=False)
    print(f"Checkpoint keys: {{list(ckpt.keys())}}")
    print(f"State dict keys (first 5): {{list(ckpt['state_dict'].keys())[:5]}}")
    print(f"Has cfg_yaml: {{'cfg_yaml' in ckpt}}")
""")

with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False) as f:
    f.write(script)
    script_path = f.name

env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = str(DEVICE)

result = subprocess.run(
    ["/home/igor/miniconda3/envs/state/bin/python", script_path],
    cwd="/home/igor/noise_scaling/modeling/STATE/state",
    env=env,
    capture_output=True, text=True,
)
os.unlink(script_path)

print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)

Token dim (emb size): 2560
Saved untrained checkpoint to /home/igor/noise_scaling/data/PBMC/46415/1.0/results/State_untrained/model/untrained.ckpt
Checkpoint keys: ['epoch', 'global_step', 'pytorch-lightning_version', 'state_dict', 'loops', 'callbacks', 'optimizer_states', 'lr_schedulers', 'hparams_name', 'hyper_parameters', 'cfg_yaml', 'protein_embeds_dict']
State dict keys (first 5): ['cls_token', 'encoder.0.weight', 'encoder.0.bias', 'encoder.1.weight', 'encoder.1.bias']
Has cfg_yaml: True



## 2. Embed test set with untrained model

In [4]:
from scaling_laws.algo import State

# Use the same State class and embed() code path as the trained model.
# Point checkpoint_dir at the untrained checkpoint location.
model_untrained_emb = State(
    base_dir=str(base_dir),
    device=DEVICE,
    max_epochs=0,
    dataset_name=DATASET,
    seed=SEED,
)

# Override paths to use the untrained results directory
model_untrained_emb.save_folder_path = base_dir / "results" / "State_untrained"
model_untrained_emb.model_name = "model"
model_untrained_emb.embeddings_path = model_untrained_emb.save_folder_path / "model" / "embeddings.csv"

# Place the untrained checkpoint where _find_best_checkpoint() will find it
model_untrained_emb.checkpoint_dir = untrained_dir
print(f"Checkpoint dir: {model_untrained_emb.checkpoint_dir}")
print(f"Checkpoint exists: {untrained_ckpt.exists()}")

embeddings_untrained = model_untrained_emb.embed()
print(f"Untrained embeddings shape: {embeddings_untrained.shape}")

Using GPU 4 (visible as cuda:0)
Checkpoint dir: /home/igor/noise_scaling/data/PBMC/46415/1.0/results/State_untrained/model
Checkpoint exists: True
  Using checkpoint: /home/igor/noise_scaling/data/PBMC/46415/1.0/results/State_untrained/model/untrained.ckpt
  Running: /home/igor/miniconda3/envs/state/bin/python -m state emb transform --checkpoint /home/igor/noise_scaling/data/PBMC/46415/1.0/results/State_untrained/model/untrained.ckpt --input /home/igor/noise_scaling/data/PBMC/test/1.0/preprocessed/preprocessed.h5ad --output /home/igor/noise_scaling/data/PBMC/46415/1.0/results/State_untrained/model/embeddings.npy


INFO:state._cli._emb._transform:Using model checkpoint: /home/igor/noise_scaling/data/PBMC/46415/1.0/results/State_untrained/model/untrained.ckpt
INFO:state._cli._emb._transform:Creating inference object
INFO:state._cli._emb._transform:Loading model from checkpoint: /home/igor/noise_scaling/data/PBMC/46415/1.0/results/State_untrained/model/untrained.ckpt
INFO:state._cli._emb._transform:Created output directory: /home/igor/noise_scaling/data/PBMC/46415/1.0/results/State_untrained/model
INFO:state._cli._emb._transform:Computing embeddings for /home/igor/noise_scaling/data/PBMC/test/1.0/preprocessed/preprocessed.h5ad
INFO:state._cli._emb._transform:Output will be saved to /home/igor/noise_scaling/data/PBMC/46415/1.0/results/State_untrained/model/embeddings.npy as a NumPy array of embeddings only (no embedded .h5ad will be written)
INFO:state.emb.inference:Auto-detected gene column: var.index (overlap: 14899/42940 protein embeddings, 71.9% of genes)
INFO:/home/igor/noise_scaling/modeling/S

!!! 14899 genes mapped to embedding file (out of 20729)


Encoding: 100%|██████████| 215/215 [01:36<00:00,  2.23it/s]
INFO:state._cli._emb._transform:Saved embeddings matrix with shape (27503, 256) to /home/igor/noise_scaling/data/PBMC/46415/1.0/results/State_untrained/model/embeddings.npy
INFO:state._cli._emb._transform:Embedding computation completed successfully!


  Embeddings: (27503, 256)
Untrained embeddings shape: (27503, 256)


## 3. Compute LMI mutual information (protein_counts)

In [5]:
# Use BaseAlgorithm's MI machinery via a State instance pointed at the untrained dir
model_untrained = State(
    base_dir=str(base_dir),
    device=DEVICE,
    max_epochs=0,
    dataset_name=DATASET,
    seed=SEED,
)

# Override paths to point to our untrained results
model_untrained.save_folder_path = base_dir / "results" / "State_untrained"
model_untrained.model_name = "model"
model_untrained.embeddings_path = untrained_dir / "embeddings.csv"

# Save embeddings as CSV (required by MI computation)
pd.DataFrame(embeddings_untrained).to_csv(model_untrained.embeddings_path, index=False)

model_untrained.signal_columns = ["protein_counts"]
mi_results = model_untrained.mutual_information(max_epochs=300)
print("\nUntrained STATE LMI results:")
for signal, mi in mi_results.items():
    print(f"  {signal}: {mi:.5f}")

Using GPU 4 (visible as cuda:0)
Using GPU for LMI computation (device 4 visible as cuda:0)
Found 4 quality-specific signal files: ['Y_celltype.l3_1.0.csv', 'Y_celltype.l3_1.0_geneformer.csv', 'Y_protein_counts_1.0.csv', 'Y_protein_counts_1.0_geneformer.csv']
using signal file /home/igor/noise_scaling/data/PBMC/test/1.0/signals/Y_celltype.l3_1.0.csv
epoch 224 (of max 300) 🌻🌻🌻🌻🌻🌻🌻 🎉🎉
success! training stopped at epoch 224
final validation loss: 1.605521749047672
State MI for Y_celltype.l3_1.0 at PBMC quality 1.0 size 46415: 0.8632999711697422
Saving results to /home/igor/noise_scaling/data/PBMC/46415/1.0/results/State_untrained/model/MI/42/Y_celltype.l3_1.0
using signal file /home/igor/noise_scaling/data/PBMC/test/1.0/signals/Y_protein_counts_1.0.csv
epoch 71 (of max 300) 🌻🌻 🎉🎉
success! training stopped at epoch 71
final validation loss: 2.1090645369361427
State MI for Y_protein_counts_1.0 at PBMC quality 1.0 size 46415: 1.1441249913298976
Saving results to /home/igor/noise_scaling/data/

## 4. Random Projection on STATE's gene set

Recompute RP using only the 14,899 genes that STATE sees (intersection of PBMC genes and ESM-2 embeddings).
This controls for the gene set difference when comparing RP vs untrained STATE.

In [6]:
import torch
from sklearn.random_projection import GaussianRandomProjection

# Load STATE's valid gene mask to get the same gene set
profile_name = f"scaling_{DATASET}_{SIZE}_{str(QUALITY).replace('.', '_')}"
state_data_dir = base_dir / "preprocessed" / "state_data"
valid_masks = torch.load(
    state_data_dir / f"valid_genes_masks_{profile_name}.torch",
    map_location="cpu", weights_only=False,
)
# Use the test split mask (same as train)
test_key = [k for k in valid_masks if "test" in k][0]
state_gene_mask = valid_masks[test_key].numpy()
n_state_genes = state_gene_mask.sum()
print(f"STATE valid gene mask: {n_state_genes} of {len(state_gene_mask)} genes")

# Load test data
test_h5ad = DATA_DIR / DATASET / "test" / str(QUALITY) / "preprocessed" / "preprocessed.h5ad"
test_adata = ad.read_h5ad(test_h5ad)
print(f"Test data: {test_adata.shape}")

# RP on STATE's gene set (same n_components=16 as original RP)
X_state_genes = test_adata.X[:, state_gene_mask].toarray()
rp_state = GaussianRandomProjection(n_components=16, random_state=42)
rp_state.fit(X_state_genes[:5, :])
X_rp_state = rp_state.transform(X_state_genes)
print(f"RP (STATE genes) embeddings: {X_rp_state.shape}")

# Save
rp_state_dir = base_dir / "results" / "RP_state_genes" / "model"
rp_state_dir.mkdir(parents=True, exist_ok=True)
rp_state_emb_path = rp_state_dir / "embeddings.csv"
pd.DataFrame(X_rp_state).to_csv(rp_state_emb_path, index=False)
print(f"Saved to {rp_state_emb_path}")

STATE valid gene mask: 14899 of 20729 genes
Test data: (27503, 20729)
RP (STATE genes) embeddings: (27503, 16)
Saved to /home/igor/noise_scaling/data/PBMC/46415/1.0/results/RP_state_genes/model/embeddings.csv


## 5. Compute LMI for RP on STATE's gene set

In [ ]:
from scaling_laws.algo import State

# Reuse BaseAlgorithm's MI machinery
model_rp_state = State(
    base_dir=str(base_dir),
    device=DEVICE,
    max_epochs=0,
    dataset_name=DATASET,
    seed=SEED,
)
model_rp_state.save_folder_path = base_dir / "results" / "RP_state_genes"
model_rp_state.model_name = "model"
model_rp_state.embeddings_path = rp_state_emb_path
model_rp_state.signal_columns = ["protein_counts"]

mi_rp_state = model_rp_state.mutual_information(max_epochs=300)
print("\nRP (STATE genes) LMI results:")
for signal, mi in mi_rp_state.items():
    print(f"  {signal}: {mi:.5f}")

Using GPU 4 (visible as cuda:0)
Using GPU for LMI computation (device 4 visible as cuda:0)
Found 4 quality-specific signal files: ['Y_celltype.l3_1.0.csv', 'Y_celltype.l3_1.0_geneformer.csv', 'Y_protein_counts_1.0.csv', 'Y_protein_counts_1.0_geneformer.csv']
using signal file /home/igor/noise_scaling/data/PBMC/test/1.0/signals/Y_celltype.l3_1.0.csv
epoch 247 (of max 300) 🌻🌻🌻🌻🌻🌻🌻🌻 🎉🎉
success! training stopped at epoch 247
final validation loss: 0.8725730250863468
State MI for Y_celltype.l3_1.0 at PBMC quality 1.0 size 46415: 2.4237353329348883
Saving results to /home/igor/noise_scaling/data/PBMC/46415/1.0/results/RP_state_genes/model/MI/42/Y_celltype.l3_1.0
using signal file /home/igor/noise_scaling/data/PBMC/test/1.0/signals/Y_protein_counts_1.0.csv
epoch 170 (of max 300) 🌻🌻🌻🌻🌻

## 5b. Random Projection on top-512 expressed genes

STATE with `pad_length=512` only sees the 512 most-expressed genes per cell.
To isolate the effect of this truncation, compute RP on the same top-512 gene set.
We use the 512 genes with highest mean expression across all test cells (a fixed set).

In [ ]:
results_root = base_dir / "results"
algos = [
    ("State",           "State (trained)"),
    ("State_untrained", "State (untrained)"),
    ("RandomProjection","RP (all 20,729 genes)"),
    ("RP_state_genes",  "RP (14,899 STATE genes)"),
    ("RP_top512_genes", "RP (top-512 genes)"),
]

rows = []
for algo_dir, algo_label in algos:
    sig = "Y_protein_counts_1.0"
    mi_base = results_root / algo_dir / "model" / "MI"
    if not mi_base.exists():
        print(f"{algo_label:35s}  NOT FOUND")
        continue
    for seed_dir in sorted(mi_base.iterdir()):
        mi_file = seed_dir / sig / "lmi_mutual_information.txt"
        if mi_file.exists():
            mi = float(mi_file.read_text().strip())
            rows.append({"Algorithm": algo_label, "dir": algo_dir, "seed": int(seed_dir.name), "LMI": mi})
            print(f"{algo_label:35s}  seed={seed_dir.name}  LMI={mi:.5f}")

scores = pd.DataFrame(rows)
scores_agg = (
    scores.groupby("Algorithm")["LMI"]
    .agg(["mean", "std", "count"])
    .rename(columns={"mean": "mean_lmi", "std": "std_lmi", "count": "n_seeds"})
    .reset_index()
    .sort_values("mean_lmi", ascending=False)
)
scores_agg["std_lmi"] = scores_agg["std_lmi"].fillna(0)
scores_agg

colors = {
    "State (trained)":         "#8172B3",
    "State (untrained)":       "#DA8BC3",
    "RP (all 20,729 genes)":   "#DD8452",
    "RP (14,899 STATE genes)": "#E8A838",
    "RP (top-512 genes)":      "#C44E52",
}

# Order: RP all genes, RP state genes, RP top-512, State untrained, State trained
plot_order = [
    "RP (all 20,729 genes)",
    "RP (14,899 STATE genes)",
    "RP (top-512 genes)",
    "State (untrained)",
    "State (trained)",
]
scores_plot = scores_agg.set_index("Algorithm").loc[
    [o for o in plot_order if o in scores_agg["Algorithm"].values]
].reset_index()

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(
    scores_plot["Algorithm"], scores_plot["mean_lmi"],
    yerr=scores_plot["std_lmi"],
    capsize=4,
    color=[colors.get(a, "#999") for a in scores_plot["Algorithm"]],
    edgecolor="black", linewidth=0.5,
)

for bar, mean, std, n in zip(
    bars, scores_plot["mean_lmi"], scores_plot["std_lmi"], scores_plot["n_seeds"]
):
    label = f"{mean:.3f}"
    if n > 1:
        label += f"\n\u00b1{std:.3f} (n={n})"
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + std + 0.02,
        label, ha="center", va="bottom", fontsize=10, fontweight="bold",
    )

ax.set_ylabel("LMI (protein_counts)", fontsize=12)
ax.set_title(
    f"PBMC {SIZE} cells, quality {QUALITY}\n"
    "Random Projection vs STATE: controlling for gene set and context window",
    fontsize=13,
)
ax.grid(axis="y", alpha=0.3)
ax.set_ylim(0, (scores_plot["mean_lmi"] + scores_plot["std_lmi"]).max() * 1.3)
plt.xticks(rotation=15, ha="right")
fig.tight_layout()
plt.show()

# Print summary
print("\nSummary:")
for _, row in scores_plot.iterrows():
    print(f"  {row['Algorithm']:35s}  LMI = {row['mean_lmi']:.4f}")

# Compute pairwise differences
vals = {row["Algorithm"]: row["mean_lmi"] for _, row in scores_plot.iterrows()}
print("\nPairwise differences:")
if "RP (all 20,729 genes)" in vals and "RP (14,899 STATE genes)" in vals:
    print(f"  Gene coverage effect (all vs STATE genes):     {vals['RP (all 20,729 genes)'] - vals['RP (14,899 STATE genes)']:+.4f}")
if "RP (14,899 STATE genes)" in vals and "RP (top-512 genes)" in vals:
    print(f"  Context window effect (14,899 vs top-512):     {vals['RP (14,899 STATE genes)'] - vals['RP (top-512 genes)']:+.4f}")
if "RP (top-512 genes)" in vals and "State (untrained)" in vals:
    print(f"  Architecture effect (RP top-512 vs untrained):  {vals['RP (top-512 genes)'] - vals['State (untrained)']:+.4f}")
if "State (untrained)" in vals and "State (trained)" in vals:
    print(f"  Training effect (untrained vs trained STATE):   {vals['State (untrained)'] - vals['State (trained)']:+.4f}")

In [ ]:
results_root = base_dir / "results"
algos = [
    ("State",           "State (trained)"),
    ("State_untrained", "State (untrained)"),
    ("RandomProjection","RP (all 20,729 genes)"),
    ("RP_state_genes",  "RP (14,899 STATE genes)"),
]

rows = []
for algo_dir, algo_label in algos:
    sig = "Y_protein_counts_1.0"
    mi_base = results_root / algo_dir / "model" / "MI"
    if not mi_base.exists():
        print(f"{algo_label:35s}  NOT FOUND")
        continue
    for seed_dir in sorted(mi_base.iterdir()):
        mi_file = seed_dir / sig / "lmi_mutual_information.txt"
        if mi_file.exists():
            mi = float(mi_file.read_text().strip())
            rows.append({"Algorithm": algo_label, "dir": algo_dir, "seed": int(seed_dir.name), "LMI": mi})
            print(f"{algo_label:35s}  seed={seed_dir.name}  LMI={mi:.5f}")

scores = pd.DataFrame(rows)
scores_agg = (
    scores.groupby("Algorithm")["LMI"]
    .agg(["mean", "std", "count"])
    .rename(columns={"mean": "mean_lmi", "std": "std_lmi", "count": "n_seeds"})
    .reset_index()
    .sort_values("mean_lmi", ascending=False)
)
scores_agg["std_lmi"] = scores_agg["std_lmi"].fillna(0)
scores_agg

In [ ]:
colors = {
    "State (trained)":         "#8172B3",
    "State (untrained)":       "#DA8BC3",
    "RP (all 20,729 genes)":   "#DD8452",
    "RP (14,899 STATE genes)": "#E8A838",
}

# Order: RP all genes, RP state genes, State untrained, State trained
plot_order = [
    "RP (all 20,729 genes)",
    "RP (14,899 STATE genes)",
    "State (untrained)",
    "State (trained)",
]
scores_plot = scores_agg.set_index("Algorithm").loc[
    [o for o in plot_order if o in scores_agg["Algorithm"].values]
].reset_index()

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(
    scores_plot["Algorithm"], scores_plot["mean_lmi"],
    yerr=scores_plot["std_lmi"],
    capsize=4,
    color=[colors.get(a, "#999") for a in scores_plot["Algorithm"]],
    edgecolor="black", linewidth=0.5,
)

for bar, mean, std, n in zip(
    bars, scores_plot["mean_lmi"], scores_plot["std_lmi"], scores_plot["n_seeds"]
):
    label = f"{mean:.3f}"
    if n > 1:
        label += f"\n\u00b1{std:.3f} (n={n})"
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + std + 0.02,
        label, ha="center", va="bottom", fontsize=10, fontweight="bold",
    )

ax.set_ylabel("LMI (protein_counts)", fontsize=12)
ax.set_title(
    f"PBMC {SIZE} cells, quality {QUALITY}\n"
    "Random Projection vs STATE: controlling for gene set",
    fontsize=13,
)
ax.grid(axis="y", alpha=0.3)
ax.set_ylim(0, (scores_plot["mean_lmi"] + scores_plot["std_lmi"]).max() * 1.3)
plt.xticks(rotation=15, ha="right")
fig.tight_layout()
plt.show()

# Print summary
print("\nSummary:")
for _, row in scores_plot.iterrows():
    print(f"  {row['Algorithm']:35s}  LMI = {row['mean_lmi']:.4f}")

# Compute differences
rp_all = scores_plot[scores_plot["Algorithm"] == "RP (all 20,729 genes)"]["mean_lmi"].values
rp_state = scores_plot[scores_plot["Algorithm"] == "RP (14,899 STATE genes)"]["mean_lmi"].values
untrained = scores_plot[scores_plot["Algorithm"] == "State (untrained)"]["mean_lmi"].values
trained = scores_plot[scores_plot["Algorithm"] == "State (trained)"]["mean_lmi"].values

if len(rp_all) and len(rp_state):
    print(f"\n  RP gene set effect (all vs STATE genes): {rp_all[0] - rp_state[0]:+.4f}")
if len(rp_state) and len(untrained):
    print(f"  Architecture effect (RP STATE genes vs untrained STATE): {rp_state[0] - untrained[0]:+.4f}")
if len(untrained) and len(trained):
    print(f"  Training effect (untrained vs trained STATE): {untrained[0] - trained[0]:+.4f}")

## 7. UMAP: Untrained vs Trained embeddings

In [ ]:
import umap

# Load trained STATE embeddings for comparison
trained_npy = base_dir / "results" / "State" / "model" / "embeddings.npy"
embeddings_trained = np.load(trained_npy)

max_cells = 5_000
rng = np.random.default_rng(42)
idx = rng.choice(embeddings_untrained.shape[0], min(max_cells, embeddings_untrained.shape[0]), replace=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, emb, title in [
    (axes[0], embeddings_untrained[idx], "Untrained STATE"),
    (axes[1], embeddings_trained[idx], "Trained STATE"),
]:
    reducer = umap.UMAP(n_components=2, random_state=42, n_jobs=1)
    coords = reducer.fit_transform(emb)
    ax.scatter(coords[:, 0], coords[:, 1], s=2, alpha=0.5, c="#888")
    ax.set_title(title)
    ax.set_xlabel("UMAP 1")
    ax.set_ylabel("UMAP 2")

fig.suptitle(f"PBMC {SIZE} — Untrained vs Trained STATE embeddings", fontsize=13)
fig.tight_layout()
plt.show()

## 8. Embedding variance check

If embeddings are collapsed, per-dimension variance will be near zero.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, emb, title in [
    (axes[0], embeddings_untrained, "Untrained STATE"),
    (axes[1], embeddings_trained, "Trained STATE"),
]:
    var = np.var(emb, axis=0)
    ax.bar(range(len(var)), np.sort(var)[::-1], width=1, alpha=0.7)
    ax.set_xlabel("Dimension (sorted)")
    ax.set_ylabel("Variance")
    ax.set_title(f"{title} — per-dim variance\ntotal var={var.sum():.2f}, effective dims={np.sum(var > 0.01 * var.max())}") 
    ax.set_yscale("log")

fig.suptitle(f"PBMC {SIZE} — Embedding variance comparison", fontsize=13)
fig.tight_layout()
plt.show()